# SurfaceTarget-CRC — phase 01 — patch 1 17

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.17
#
# TCGA RNA-SEQ GENE EXPRESSION MATRIX CONSTRUCTION
#
# Objectives:
# 1. Load all verified STAR Counts files
# 2. Extract unstranded raw gene counts
# 3. Validate gene identifiers and annotations
# 4. Construct a unified gene-by-sample count matrix
# 5. Preserve sample and patient metadata
# 6. Calculate preliminary expression QC metrics
# 7. Save compressed expression data
#
# IMPORTANT:
# No gene filtering.
# No normalization.
# No differential expression analysis.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import hashlib
import json
import shutil

# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

PROCESSED = ROOT / "data/processed"

RESULTS = ROOT / "results/tables"

PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

TUMOR_MANIFEST = (
    META / "tcga_selected_tumor_rnaseq_manifest.csv"
)

NORMAL_MANIFEST = (
    META / "tcga_normal_rnaseq_manifest.csv"
)

VERIFICATION_PATH = (
    META / "tcga_rnaseq_download_verification.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.17 — EXPRESSION MATRIX CONSTRUCTION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load sample manifests
# ------------------------------------------------------------

tumor = pd.read_csv(
    TUMOR_MANIFEST,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_MANIFEST,
    dtype=str,
    keep_default_na=False
)

tumor["dataset_group"] = "Tumor"
normal["dataset_group"] = "Normal"

manifest = pd.concat(
    [tumor, normal],
    ignore_index=True
)

if len(tumor) != 421:
    raise RuntimeError("Unexpected tumor sample count.")

if len(normal) != 51:
    raise RuntimeError("Unexpected normal sample count.")

if manifest["file_id"].duplicated().any():
    raise RuntimeError("Duplicate file IDs detected.")

print("\nSample manifests loaded.")

print("Tumor samples:", len(tumor))
print("Normal samples:", len(normal))
print("Total samples:", len(manifest))


# ------------------------------------------------------------
# 3. Verify previous download status
# ------------------------------------------------------------

verification = pd.read_csv(
    VERIFICATION_PATH,
    dtype=str
)

verified_ids = set(
    verification.loc[
        verification["verified"]
        .astype(str)
        .str.lower()
        .eq("true"),
        "file_id"
    ]
)

manifest_ids = set(manifest["file_id"])

if verified_ids != manifest_ids:
    raise RuntimeError(
        "Download verification records do not match "
        "the current manifest."
    )

print("\nDownload verification records: PASSED")


# ------------------------------------------------------------
# 4. Define STAR summary rows
# ------------------------------------------------------------

STAR_SUMMARY = {
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
}

REQUIRED_COLUMNS = {
    "gene_id",
    "gene_name",
    "gene_type",
    "unstranded"
}


# ------------------------------------------------------------
# 5. STAR Counts file parser
# ------------------------------------------------------------

def load_gene_counts(file_path):

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False,
        dtype={"gene_id": str}
    )

    missing = REQUIRED_COLUMNS - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Missing required columns in {file_path}: "
            f"{missing}"
        )

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY)
    ].copy()

    if set(summary["gene_id"]) != STAR_SUMMARY:
        raise RuntimeError(
            f"Incomplete STAR summary: {file_path}"
        )

    if len(summary) != 4:
        raise RuntimeError(
            f"Unexpected STAR summary rows: {file_path}"
        )

    if genes.empty:
        raise RuntimeError(
            f"No gene counts found: {file_path}"
        )

    if genes["gene_id"].isna().any():
        raise RuntimeError(
            f"Missing gene IDs: {file_path}"
        )

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            f"Duplicate gene IDs: {file_path}"
        )

    counts = pd.to_numeric(
        genes["unstranded"],
        errors="raise"
    )

    if counts.isna().any():
        raise RuntimeError(
            f"Missing counts: {file_path}"
        )

    if (counts < 0).any():
        raise RuntimeError(
            f"Negative counts: {file_path}"
        )

    if not np.all(
        counts.to_numpy() == np.floor(counts.to_numpy())
    ):
        raise RuntimeError(
            f"Non-integer raw counts: {file_path}"
        )

    genes["unstranded"] = counts.astype(np.int64)

    return genes, summary


# ------------------------------------------------------------
# 6. Initialize reference gene annotations
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("INITIALIZING GENE REFERENCE")
print("-" * 65)

first_file = Path(
    manifest.iloc[0]["local_path"]
)

if not first_file.exists():
    raise FileNotFoundError(first_file)

reference_genes, _ = load_gene_counts(first_file)

gene_metadata = reference_genes[
    ["gene_id", "gene_name", "gene_type"]
].copy()

gene_metadata = gene_metadata.fillna("")

reference_ids = (
    gene_metadata["gene_id"]
    .astype(str)
    .to_numpy()
)

reference_gene_types = (
    gene_metadata["gene_type"]
    .astype(str)
    .to_numpy()
)

reference_index = pd.Index(reference_ids)

reference_gene_set = set(reference_ids)

n_genes = len(reference_ids)
n_samples = len(manifest)

print("\nReference file:", first_file.name)

print("Total genes:", n_genes)

print(
    "Protein-coding genes:",
    int(
        (
            gene_metadata["gene_type"]
            == "protein_coding"
        ).sum()
    )
)

print("\nGene reference initialized.")


# ------------------------------------------------------------
# 7. Allocate expression matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ALLOCATING EXPRESSION MATRIX")
print("-" * 65)

# Rows = genes
# Columns = samples
#
# int64 preserves integer count values.

count_matrix = np.empty(
    (n_genes, n_samples),
    dtype=np.int64
)

matrix_memory_mb = (
    count_matrix.nbytes / (1024 ** 2)
)

print(
    f"\nMatrix dimensions: "
    f"{n_genes:,} genes x {n_samples:,} samples"
)

print(
    f"Allocated matrix memory: "
    f"{matrix_memory_mb:.2f} MiB"
)


# ------------------------------------------------------------
# 8. Process all RNA-seq files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CONSTRUCTING GENE EXPRESSION MATRIX")
print("-" * 65)

qc_records = []

reordered_files = 0

for column_index, row in manifest.iterrows():

    file_id = row["file_id"]

    file_path = Path(row["local_path"])

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing RNA-seq file: {file_path}"
        )

    expected_size = int(row["file_size_bytes"])

    if file_path.stat().st_size != expected_size:
        raise RuntimeError(
            f"File size changed: {file_id}"
        )

    genes, summary = load_gene_counts(file_path)

    # --------------------------------------------
    # Validate gene identifiers
    # --------------------------------------------

    current_ids = (
        genes["gene_id"]
        .astype(str)
        .to_numpy()
    )

    if len(current_ids) != n_genes:
        raise RuntimeError(
            f"Gene count mismatch: {file_id}"
        )

    if not np.array_equal(
        current_ids,
        reference_ids
    ):

        if set(current_ids) != reference_gene_set:
            raise RuntimeError(
                f"Gene identifier mismatch: {file_id}"
            )

        # Same genes, different row order.
        # Reorder to the reference gene order.

        genes = (
            genes.set_index("gene_id")
            .loc[reference_index]
            .reset_index()
        )

        reordered_files += 1

    # --------------------------------------------
    # Validate gene-type annotations
    # --------------------------------------------

    current_types = (
        genes["gene_type"]
        .fillna("")
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        current_types,
        reference_gene_types
    ):
        raise RuntimeError(
            f"Gene-type annotation mismatch: {file_id}"
        )

    # --------------------------------------------
    # Store raw counts
    # --------------------------------------------

    counts = (
        genes["unstranded"]
        .to_numpy(dtype=np.int64)
    )

    count_matrix[:, column_index] = counts

    # --------------------------------------------
    # Calculate preliminary QC metrics
    # --------------------------------------------

    assigned_counts = int(counts.sum())

    detected_genes = int(
        np.count_nonzero(counts)
    )

    detected_protein_coding = int(
        np.count_nonzero(
            counts[
                reference_gene_types == "protein_coding"
            ]
        )
    )

    summary_values = dict(
        zip(
            summary["gene_id"],
            pd.to_numeric(
                summary["unstranded"],
                errors="raise"
            )
        )
    )

    total_counted = (
        assigned_counts
        + sum(summary_values.values())
    )

    assignment_fraction = (
        assigned_counts / total_counted
        if total_counted > 0
        else np.nan
    )

    qc_records.append({

        "file_id": file_id,

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "project": row["project"],

        "dataset_group": row["dataset_group"],

        "cohort_tier": row["cohort_tier"],

        "workflow_version": row["workflow_version"],

        "assigned_gene_counts": assigned_counts,

        "detected_genes": detected_genes,

        "detected_protein_coding":
            detected_protein_coding,

        "assignment_fraction":
            assignment_fraction,

        "N_unmapped":
            int(summary_values["N_unmapped"]),

        "N_multimapping":
            int(summary_values["N_multimapping"]),

        "N_noFeature":
            int(summary_values["N_noFeature"]),

        "N_ambiguous":
            int(summary_values["N_ambiguous"])
    })

    if (column_index + 1) % 25 == 0:

        print(
            f"Processed {column_index + 1} "
            f"of {n_samples} files"
        )

print(
    f"\nProcessed {n_samples} of {n_samples} files"
)

print("\nMatrix construction complete.")


# ------------------------------------------------------------
# 9. Validate expression matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXPRESSION MATRIX VALIDATION")
print("-" * 65)

if count_matrix.shape != (
    n_genes,
    n_samples
):
    raise RuntimeError(
        "Expression matrix dimensions incorrect."
    )

if np.any(count_matrix < 0):
    raise RuntimeError(
        "Negative values detected in count matrix."
    )

if len(qc_records) != n_samples:
    raise RuntimeError(
        "QC record count mismatch."
    )

# Confirm that matrix column sums match
# independently recorded library sizes.

qc = pd.DataFrame(qc_records)

matrix_library_sizes = (
    count_matrix.sum(axis=0)
)

recorded_library_sizes = (
    qc["assigned_gene_counts"]
    .to_numpy(dtype=np.int64)
)

if not np.array_equal(
    matrix_library_sizes,
    recorded_library_sizes
):
    raise RuntimeError(
        "Library-size validation failed."
    )

print("\nMatrix dimensions: PASSED")

print("Nonnegative raw counts: PASSED")

print("Library-size consistency: PASSED")

print("Sample metadata correspondence: PASSED")

print(
    "Files requiring gene-order correction:",
    reordered_files
)


# ------------------------------------------------------------
# 10. Preliminary expression QC summary
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PRELIMINARY EXPRESSION QC")
print("-" * 65)

qc["library_size_millions"] = (
    qc["assigned_gene_counts"] / 1e6
)

qc["assignment_percent"] = (
    qc["assignment_fraction"] * 100
)

qc_summary = (
    qc.groupby(
        [
            "project",
            "dataset_group",
            "cohort_tier"
        ]
    )
    .agg(
        samples=("file_id", "count"),

        median_library_size_millions=(
            "library_size_millions",
            "median"
        ),

        median_detected_genes=(
            "detected_genes",
            "median"
        ),

        median_detected_protein_coding=(
            "detected_protein_coding",
            "median"
        ),

        median_assignment_percent=(
            "assignment_percent",
            "median"
        )
    )
    .reset_index()
)

print("\n")
print(
    qc_summary.round(2).to_string(index=False)
)


# ------------------------------------------------------------
# 11. Save gene annotation metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PROCESSED EXPRESSION DATA")
print("-" * 65)

GENE_METADATA_PATH = (
    META / "tcga_gene_annotations.csv"
)

SAMPLE_METADATA_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

QC_SUMMARY_PATH = (
    RESULTS / "tcga_expression_initial_qc_summary.csv"
)

gene_metadata.to_csv(
    GENE_METADATA_PATH,
    index=False
)

# Matrix column order is exactly the
# order of rows in this sample metadata file.

sample_metadata = manifest.copy()

sample_metadata["matrix_column_index"] = (
    np.arange(n_samples)
)

sample_metadata.to_csv(
    SAMPLE_METADATA_PATH,
    index=False
)

qc.to_csv(
    QC_PATH,
    index=False
)

qc_summary.to_csv(
    QC_SUMMARY_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Save compressed raw-count matrix
# ------------------------------------------------------------

MATRIX_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.npz"
)

np.savez_compressed(

    MATRIX_PATH,

    counts=count_matrix,

    gene_ids=reference_ids,

    file_ids=manifest["file_id"].to_numpy()
)

print("\nCompressed count matrix saved.")


# ------------------------------------------------------------
# 13. Verify saved matrix
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVED MATRIX VERIFICATION")
print("-" * 65)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as saved:

    saved_shape = saved["counts"].shape

    saved_gene_ids = saved["gene_ids"]

    saved_file_ids = saved["file_ids"]

    first_column_matches = np.array_equal(
        saved["counts"][:, 0],
        count_matrix[:, 0]
    )

if saved_shape != count_matrix.shape:
    raise RuntimeError(
        "Saved matrix dimensions are incorrect."
    )

if not np.array_equal(
    saved_gene_ids,
    reference_ids
):
    raise RuntimeError(
        "Saved gene identifiers do not match."
    )

if not np.array_equal(
    saved_file_ids,
    manifest["file_id"].to_numpy()
):
    raise RuntimeError(
        "Saved sample identifiers do not match."
    )

if not first_column_matches:
    raise RuntimeError(
        "Saved count values failed verification."
    )

print("\nSaved matrix verification: PASSED")


# ------------------------------------------------------------
# 14. Calculate file checksum
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


matrix_sha256 = sha256_file(
    MATRIX_PATH
)

matrix_size_mb = (
    MATRIX_PATH.stat().st_size
    / (1024 ** 2)
)


# ------------------------------------------------------------
# 15. Save processing provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.17",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "count_column": "unstranded",

    "matrix_orientation": "genes_by_samples",

    "n_genes": int(n_genes),

    "n_samples": int(n_samples),

    "n_tumor_samples": int(len(tumor)),

    "n_normal_samples": int(len(normal)),

    "data_type": "int64",

    "gene_filtering_performed": False,

    "normalization_performed": False,

    "differential_expression_performed": False,

    "files_reordered_to_reference": int(
        reordered_files
    ),

    "matrix_file": str(MATRIX_PATH),

    "matrix_sha256": matrix_sha256,

    "matrix_size_mb": float(matrix_size_mb),

    "validation": [
        "Gene identifier consistency",
        "Gene-type annotation consistency",
        "Nonnegative integer counts",
        "Matrix dimensions",
        "Library-size consistency",
        "Saved matrix identifiers",
        "Saved matrix first-column values"
    ],

    "limitations": [
        "Full-cohort quality control remains pending.",
        "Workflow-version confounding remains unresolved.",
        "No biological conclusions have been drawn.",
        "Normal samples are not all paired with tumor samples.",
        "Tier B is computationally low-MSI, not confirmed MSS."
    ]
}

PROVENANCE_PATH = (
    META /
    "tcga_expression_matrix_provenance.json"
)

with open(
    PROVENANCE_PATH,
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 16. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.17 SUMMARY")
print("=" * 65)

print("\nExpression matrix dimensions:")

print(
    f"{n_genes:,} genes x {n_samples:,} samples"
)

print("\nTumor samples:", len(tumor))

print("Normal samples:", len(normal))

print(
    "\nCompressed matrix size:",
    f"{matrix_size_mb:.2f} MiB"
)

print("\nSaved files:")

print("1. tcga_unstranded_raw_counts.npz")

print("2. tcga_gene_annotations.csv")

print("3. tcga_expression_sample_metadata.csv")

print("4. tcga_expression_initial_qc.csv")

print("5. tcga_expression_initial_qc_summary.csv")

print("6. tcga_expression_matrix_provenance.json")

print("\nAll matrix validation checks passed.")

print("\nNo normalization performed.")

print("No genes or samples excluded.")

print("\nPATCH 1.17 COMPLETE")

print("=" * 65)

In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PATCH 1.17 — MATRIX SERIALIZATION REPAIR
#
# Fix:
# Convert gene IDs and file IDs to Unicode string arrays.
#
# IMPORTANT:
# Does NOT reprocess the 472 RNA-seq files.
# Does NOT download additional data.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import hashlib
import json
import os

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.17 — MATRIX SERIALIZATION REPAIR")
print("=" * 65)

# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"

MATRIX_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.npz"
)

REPAIRED_PATH = (
    PROCESSED /
    "tcga_unstranded_raw_counts.repaired.npz"
)

GENES_PATH = META / "tcga_gene_annotations.csv"

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = (
    RESULTS / "tcga_expression_initial_qc.csv"
)

PROVENANCE_PATH = (
    META / "tcga_expression_matrix_provenance.json"
)

# ------------------------------------------------------------
# 2. Validate required files
# ------------------------------------------------------------

required_files = [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    QC_PATH
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Required file missing: {path}"
        )

print("\nAll required files found.")

# ------------------------------------------------------------
# 3. Recover existing raw-count matrix
# ------------------------------------------------------------

print("\nRecovering previously saved count matrix...")

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as archive:

    # Only load numeric counts.
    # The original object-based identifiers
    # are deliberately not loaded.

    counts = archive["counts"]

print("\nCount matrix recovered successfully.")

print("Dimensions:", counts.shape)

print("Data type:", counts.dtype)

if counts.shape != (60660, 472):
    raise RuntimeError(
        "Unexpected count matrix dimensions."
    )

if not np.issubdtype(counts.dtype, np.integer):
    raise RuntimeError(
        "Raw-count matrix is not integer-valued."
    )

if np.any(counts < 0):
    raise RuntimeError(
        "Negative count values detected."
    )

# ------------------------------------------------------------
# 4. Recover gene and sample identifiers
# ------------------------------------------------------------

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(QC_PATH)

# Explicit Unicode conversion fixes
# the object-array serialization problem.

gene_ids = np.asarray(
    genes["gene_id"].tolist(),
    dtype=np.str_
)

file_ids = np.asarray(
    samples["file_id"].tolist(),
    dtype=np.str_
)

if len(gene_ids) != counts.shape[0]:
    raise RuntimeError(
        "Gene identifier count mismatch."
    )

if len(file_ids) != counts.shape[1]:
    raise RuntimeError(
        "Sample identifier count mismatch."
    )

if len(set(gene_ids)) != len(gene_ids):
    raise RuntimeError(
        "Duplicate gene identifiers detected."
    )

if len(set(file_ids)) != len(file_ids):
    raise RuntimeError(
        "Duplicate file identifiers detected."
    )

# Ensure sample metadata preserves matrix order.

column_indices = pd.to_numeric(
    samples["matrix_column_index"],
    errors="raise"
).to_numpy(dtype=np.int64)

if not np.array_equal(
    column_indices,
    np.arange(len(samples))
):
    raise RuntimeError(
        "Sample column order is inconsistent."
    )

# Compare with recorded QC library sizes.

if not np.array_equal(
    qc["file_id"].astype(str).to_numpy(),
    file_ids
):
    raise RuntimeError(
        "QC sample order does not match the matrix."
    )

expected_library_sizes = pd.to_numeric(
    qc["assigned_gene_counts"],
    errors="raise"
).to_numpy(dtype=np.int64)

actual_library_sizes = counts.sum(axis=0)

if not np.array_equal(
    expected_library_sizes,
    actual_library_sizes
):
    raise RuntimeError(
        "Library-size validation failed."
    )

print("\nGene identifier validation: PASSED")
print("Sample identifier validation: PASSED")
print("Sample column order: PASSED")
print("Library-size validation: PASSED")

# ------------------------------------------------------------
# 5. Save corrected archive
# ------------------------------------------------------------

print("\nSaving repaired expression matrix...")

np.savez_compressed(
    REPAIRED_PATH,
    counts=counts,
    gene_ids=gene_ids,
    file_ids=file_ids
)

# ------------------------------------------------------------
# 6. Verify repaired archive
# ------------------------------------------------------------

print("\nVerifying repaired matrix...")

with np.load(
    REPAIRED_PATH,
    allow_pickle=False
) as saved:

    saved_counts = saved["counts"]

    saved_gene_ids = saved["gene_ids"]

    saved_file_ids = saved["file_ids"]

    if not np.array_equal(
        saved_counts,
        counts
    ):
        raise RuntimeError(
            "Saved count values do not match."
        )

    if not np.array_equal(
        saved_gene_ids,
        gene_ids
    ):
        raise RuntimeError(
            "Saved gene identifiers do not match."
        )

    if not np.array_equal(
        saved_file_ids,
        file_ids
    ):
        raise RuntimeError(
            "Saved sample identifiers do not match."
        )

    if saved_gene_ids.dtype.kind != "U":
        raise RuntimeError(
            "Gene IDs were not saved as Unicode strings."
        )

    if saved_file_ids.dtype.kind != "U":
        raise RuntimeError(
            "File IDs were not saved as Unicode strings."
        )

print("\nRepaired matrix verification: PASSED")

# ------------------------------------------------------------
# 7. Replace original archive
# ------------------------------------------------------------

# Replace only after all verification checks pass.

os.replace(
    REPAIRED_PATH,
    MATRIX_PATH
)

print("\nOriginal archive replaced with verified version.")

# ------------------------------------------------------------
# 8. Calculate SHA256 checksum
# ------------------------------------------------------------

sha256 = hashlib.sha256()

with open(MATRIX_PATH, "rb") as f:

    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):

        sha256.update(chunk)

matrix_sha256 = sha256.hexdigest()

matrix_size_mib = (
    MATRIX_PATH.stat().st_size
    / (1024 ** 2)
)

# ------------------------------------------------------------
# 9. Save processing provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.17",

    "repair": "Unicode identifier serialization",

    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI Genomic Data Commons",

    "workflow": "STAR - Counts",

    "count_column": "unstranded",

    "matrix_orientation": "genes_by_samples",

    "n_genes": int(len(gene_ids)),

    "n_samples": int(len(file_ids)),

    "n_tumor_samples": 421,

    "n_normal_samples": 51,

    "data_type": str(counts.dtype),

    "identifier_data_type": "Unicode strings",

    "gene_filtering_performed": False,

    "normalization_performed": False,

    "differential_expression_performed": False,

    "matrix_file": str(MATRIX_PATH),

    "matrix_sha256": matrix_sha256,

    "matrix_size_mib": float(matrix_size_mib),

    "validation": [
        "Gene identifier count",
        "Sample identifier count",
        "Unique gene identifiers",
        "Unique sample identifiers",
        "Sample column order",
        "Raw integer counts",
        "Nonnegative counts",
        "All sample library sizes",
        "Complete saved matrix equality",
        "Saved gene identifier equality",
        "Saved file identifier equality",
        "Unicode identifier serialization"
    ],

    "limitations": [
        "Full-cohort expression QC remains pending.",
        "Workflow-version confounding remains unresolved.",
        "Normal tissue controls are not all patient-matched.",
        "Tier B is computationally low-MSI, not confirmed MSS.",
        "No biological conclusions have been drawn."
    ]
}

with open(PROVENANCE_PATH, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.17 — FINAL SUMMARY")
print("=" * 65)

print("\nExpression matrix:")

print(
    f"{len(gene_ids):,} genes x "
    f"{len(file_ids):,} samples"
)

print("\nTumor samples: 421")
print("Normal samples: 51")

print(
    f"\nCompressed matrix size: "
    f"{matrix_size_mib:.2f} MiB"
)

print("\nSaved matrix verification: PASSED")

print("Gene identifiers: VERIFIED")

print("Sample identifiers: VERIFIED")

print("Library sizes: VERIFIED")

print("Raw count values: VERIFIED")

print("\nProvenance: SAVED")

print("\nNo RNA-seq files reprocessed.")

print("No additional data downloaded.")

print("\nPATCH 1.17 COMPLETE")

print("=" * 65)